# Train LEGO YOLOv8 Detector

This notebook fine-tunes YOLOv8 on a LEGO brick detection dataset using a free Colab GPU.

**Steps:**
1. Get a free Roboflow API key at https://app.roboflow.com (Settings → API Key)
2. Paste your API key in the cell below
3. Runtime → Run all
4. Download `lego_yolov8.pt` when training completes (~15-20 min)
5. Place it in your `lego_classifier/` project root

**Important:** Make sure you're using a GPU runtime:
- Runtime → Change runtime type → T4 GPU

In [ ]:
# ============================================
# PASTE YOUR ROBOFLOW API KEY HERE
# ============================================
ROBOFLOW_API_KEY = ""  # <-- paste between the quotes

assert ROBOFLOW_API_KEY, "Please paste your Roboflow API key above!"

In [ ]:
# Verify GPU is available
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND'}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_global_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Install dependencies
!pip install -q ultralytics roboflow

In [ ]:
# Download LEGO detection dataset from Roboflow
from roboflow import Roboflow
import os, yaml

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Try several known LEGO detection datasets until one works.
DATASETS = [
    ("lego-brick-recognition", "lego-brick-recognition1", 1),
    ("research-g0ycu", "lego-brick-object-detection-40", 1),
    ("legos-3oate", "lego-detection-fsxai", 1),
    ("legodataset-0rrtg", "lego-detection", 1),
    ("aalborg-university-m8ved", "lego-briks-detection-yolo", 1),
    ("public-xzxhu", "lego-dftae", 1),
    ("lego-lo035", "brick-detection-xlifo", 1),
    ("sergen-gnen", "lego-detector-quixk", 2),
]

dataset = None
for ws, proj, ver in DATASETS:
    try:
        print(f"Trying: {ws}/{proj} v{ver} ...")
        project = rf.workspace(ws).project(proj)
        version = project.version(ver)
        dataset = version.download("yolov8", location="/content/lego_dataset")
        print(f"SUCCESS: Downloaded {ws}/{proj}")
        break
    except Exception as e:
        print(f"  Failed: {e}")
        continue

if dataset is None:
    print("\n" + "="*60)
    print("None of the preset datasets worked.")
    print("Go to https://universe.roboflow.com and search 'lego brick detection'")
    print("Pick a dataset, click 'Download Dataset' → YOLOv8 → 'Show download code'")
    print("Copy the snippet and run it in the next cell.")
    print("="*60)
else:
    # Remap all classes to a single "lego" class so YOLO just detects
    # any LEGO piece. Part identification is CLIP's job, not YOLO's.
    data_yaml_path = os.path.join(dataset.location, "data.yaml")
    with open(data_yaml_path, "r") as f:
        data_cfg = yaml.safe_load(f)

    original_classes = data_cfg.get("names", {})
    num_original = len(original_classes)
    print(f"\nOriginal classes ({num_original}): {original_classes}")

    if num_original > 1:
        print("Remapping all classes → single 'lego' class for better generalization...")

        # Update data.yaml
        data_cfg["names"] = {0: "lego"}
        data_cfg["nc"] = 1
        with open(data_yaml_path, "w") as f:
            yaml.dump(data_cfg, f)

        # Remap all label files: replace class index with 0
        for split in ["train", "valid", "test"]:
            labels_dir = os.path.join(dataset.location, split, "labels")
            if not os.path.isdir(labels_dir):
                continue
            for label_file in os.listdir(labels_dir):
                if not label_file.endswith(".txt"):
                    continue
                filepath = os.path.join(labels_dir, label_file)
                with open(filepath, "r") as f:
                    lines = f.readlines()
                with open(filepath, "w") as f:
                    for line in lines:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            parts[0] = "0"  # remap class to 0
                            f.write(" ".join(parts) + "\n")

        print("Done! All classes remapped to single 'lego' class.")
    else:
        print("Already single-class — no remapping needed.")

    print(f"\nDataset location: {dataset.location}")

In [ ]:
# Check dataset structure
!cat /content/lego_dataset/data.yaml
print("\n--- Train images ---")
!ls /content/lego_dataset/train/images | head -5
print(f"Total: {len(__import__('os').listdir('/content/lego_dataset/train/images'))} images")

In [ ]:
# Train YOLOv8m on the LEGO dataset
from ultralytics import YOLO

model = YOLO("yolov8m.pt")  # start from COCO pretrained weights

results = model.train(
    data="/content/lego_dataset/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    name="lego_detector",
    patience=10,      # early stop if no improvement for 10 epochs
    save=True,
    plots=True,
    verbose=True,
)

print(f"\nTraining complete! Results saved to: {results.save_dir}")

In [ ]:
# Show training results
from IPython.display import Image, display
from pathlib import Path

save_dir = Path(results.save_dir)

# Show confusion matrix
if (save_dir / "confusion_matrix.png").exists():
    display(Image(filename=str(save_dir / "confusion_matrix.png"), width=600))

# Show results plot
if (save_dir / "results.png").exists():
    display(Image(filename=str(save_dir / "results.png"), width=800))

# Show sample predictions
if (save_dir / "val_batch0_pred.jpg").exists():
    print("\nSample predictions on validation set:")
    display(Image(filename=str(save_dir / "val_batch0_pred.jpg"), width=800))

In [ ]:
# Validate the trained model
best_model = YOLO(str(save_dir / "weights" / "best.pt"))
metrics = best_model.val(data="/content/lego_dataset/data.yaml")

print(f"\nmAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

In [ ]:
# Copy best weights to an easy download location
import shutil

output_path = "/content/lego_yolov8.pt"
shutil.copy2(save_dir / "weights" / "best.pt", output_path)

size_mb = Path(output_path).stat().st_size / 1024 / 1024
print(f"Model saved to: {output_path} ({size_mb:.1f} MB)")
print(f"\n" + "="*60)
print(f"Download lego_yolov8.pt from the file browser on the left")
print(f"(click the folder icon, then right-click → Download)")
print(f"")
print(f"Place it in your lego_classifier/ project root folder.")
print(f"The server will auto-detect and use it on next startup.")
print(f"="*60)

In [ ]:
# Optional: download directly from Colab
from google.colab import files
files.download("/content/lego_yolov8.pt")